# Colab: real-data PINN, batch 4 (final runs)

**Settings are now fixed** (chosen in batch 3 on validation, by rules set in advance):
- canal seepage on (GEC-2015 norm);
- zoned design = **shared** (one water-level network for the whole area, storage / transmissivity / pumping per zone).

This batch trains the two models that are compared in the report, each with **5 seeds × 3 block layouts**:
- `regional`: one PINN for the whole area (the main comparison);
- `shared`: our zoned PINN.

Layout 0 / seed 0 of both already exists (batch 3), so there are **28 runs**.
They go in rounds: each round is one seed of both models on all 3 layouts (6 runs, about 2.5 hours),
so a batch stopped part-way is still a fair comparison.

Training scores **validation** only. The test wells stay sealed; Claude scores the test set once, after all runs are in.

**Before you start:**
1. Put the newest `pinn_code.zip` in `MyDrive/pinn/`.
2. **Runtime → Change runtime type → T4 GPU**, then **Runtime → Run all**.

**Timing.** About 11–13 hours in total (regional runs about 20 minutes, shared about 25–35), so this will take
2–3 Colab sessions. Keep the laptop awake. When Colab disconnects or hits its daily limit, just **Run all** again
later: finished runs are skipped. Run the last cell whenever you stop, and send the zip.

In [ ]:
LAYOUTS = [0, 1, 2]
SEEDS = [0, 1, 2, 3, 4]
DRIVE_ROOT = "/content/drive/MyDrive/pinn"

In [ ]:
import glob
import os
import shutil
import zipfile
from google.colab import drive

drive.mount("/content/drive")
os.makedirs(f"{DRIVE_ROOT}/runs/logs", exist_ok=True)
CODE = "/content/pinn"
shutil.rmtree(CODE, ignore_errors=True)
with zipfile.ZipFile(f"{DRIVE_ROOT}/pinn_code.zip") as z:
    z.extractall(CODE)
%cd {CODE}

In [ ]:
# PyTorch is already installed on Colab; this adds everything else (about 1 minute).
!pip install -q -r requirements.txt
!pip install -q -e .
os.environ["PINN_RUNS_DIR"] = f"{DRIVE_ROOT}/runs"
!python scripts/check_env.py
!python -m pytest -q -m "not data"

## Runs

In [ ]:
def done(pattern):
    return bool(glob.glob(f"{DRIVE_ROOT}/runs/{pattern}/metrics.json"))

LOG = f"{DRIVE_ROOT}/runs/logs/real4.log"
MODELS = [("kmeansk5_shared", "--shared"), ("regional", "--regional")]
for seed in SEEDS:
    for layout in LAYOUTS:
        for name, flag in MODELS:
            if layout == 0 and seed == 0:
                continue   # done in batch 3
            if done(f"*_real_L{layout}_{name}_s{seed}_tbc"):
                print("already done:", name, layout, seed)
                continue
            !python -u scripts/run_real.py {flag} --layout {layout} --seed {seed} --tag tbc --set canal_seepage.strength=1 --stage-only val 2>&1 | tee -a {LOG}

## Bundle the results for Claude
Makes `MyDrive/pinn/real4_results.zip` with every **finished** final run (run it any time).
Download it, put it in the `pinn` folder on the laptop, then tell Claude.

In [ ]:
out = f"{DRIVE_ROOT}/real4_results.zip"
n = 0
with zipfile.ZipFile(out, "w", zipfile.ZIP_DEFLATED) as z:
    for name, _ in MODELS:
        for run in glob.glob(f"{DRIVE_ROOT}/runs/*_real_L*_{name}_s*_tbc"):
            if not os.path.exists(f"{run}/metrics.json"):
                continue
            n += 1
            for path in glob.glob(f"{run}/*"):
                z.write(path, os.path.relpath(path, f"{DRIVE_ROOT}/runs"))
    if os.path.exists(LOG):
        z.write(LOG, "logs/real4.log")
print("wrote", out, f"({n} finished runs, {os.path.getsize(out) / 1e6:.2f} MB)")